# Smart Traffic Light Police-Style Full Training

Notebook này train lại từ đầu, không fine-tune từ `dqn_eval_best.pt`. Mục tiêu là policy có khả năng điều tiết giống cảnh sát giao thông: ưu tiên luồng đông nhưng không bỏ đói pha còn lại, cân bằng waiting time, queue, throughput và starvation safety.

## 1. Install dependencies and prepare repo

Chạy cell clone nếu Kaggle session chưa có source repo. Nếu notebook đã nằm trong repo, cell này chỉ cài dependency và kiểm tra đường dẫn.

In [ ]:
!pip install -q eclipse-sumo traci sumolib libsumo gymnasium pydantic pydantic-settings tqdm

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/dev-truong/smart-traffic-system.git"
REPO_DIR = Path("/kaggle/working/smart-traffic-system")

if not Path("common/constants.py").exists():
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

sys.path.insert(0, str(Path.cwd()))
print("Repo:", Path.cwd())

## 2. Configure SUMO and verify action space

In [ ]:
import sumo

os.environ["SUMO_HOME"] = os.path.dirname(sumo.__file__)
print("SUMO_HOME =", os.environ["SUMO_HOME"])
!sumo --version

from common.constants import NUM_ACTIONS, PHASE_DIRECTIONS, PhaseAction

assert NUM_ACTIONS == 2
assert set(PhaseAction) == {PhaseAction.EAST_WEST, PhaseAction.NORTH_SOUTH}
assert len(PHASE_DIRECTIONS[PhaseAction.EAST_WEST]) == 2
assert len(PHASE_DIRECTIONS[PhaseAction.NORTH_SOUTH]) == 2
print("Verified 2-phase action space:", PHASE_DIRECTIONS)

## 3. Build scenario bank

Training dùng interleaved scenarios ngay từ episode 1 để tránh catastrophic forgetting. Scenario EW và NS được cân bằng đối xứng.

In [ ]:
!python -m simulation.net.build_net
!python -m simulation.net.generate_routes --duration 1800 --seed 42 --out simulation/net/intersection.rou.xml

from copy import deepcopy
from pathlib import Path
import xml.etree.ElementTree as ET


def flow_origin(flow):
    route = flow.get("route", "")
    if route.startswith("route_"):
        return route.split("_")[1]
    parts = flow.get("id", "").split("_")
    return parts[-2] if len(parts) >= 2 else ""


def write_sumocfg(path, route_file):
    path.write_text(f'''<?xml version="1.0" encoding="UTF-8"?>
<configuration>
    <input>
        <net-file value="intersection.net.xml"/>
        <route-files value="{route_file}"/>
        <additional-files value="vtypes.add.xml"/>
    </input>
    <time>
        <begin value="0"/>
        <step-length value="1"/>
    </time>
    <processing>
        <time-to-teleport value="-1"/>
    </processing>
    <report>
        <no-step-log value="true"/>
        <duration-log.disable value="true"/>
    </report>
</configuration>
''', encoding="utf-8")


def scaled_rate(flow, *, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    origin = flow_origin(flow)
    axis_scale = ew_scale if origin in {"E", "W"} else ns_scale
    return float(flow.get("vehsPerHour")) * all_scale * axis_scale


def make_scaled_scenario(name, *, duration_s=1800, all_scale=1.0, ew_scale=1.0, ns_scale=1.0):
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    tree = ET.parse(src)
    root = tree.getroot()
    for flow in root.findall("flow"):
        flow.set("end", str(duration_s))
        flow.set("vehsPerHour", f"{scaled_rate(flow, all_scale=all_scale, ew_scale=ew_scale, ns_scale=ns_scale):.2f}")
    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    write_sumocfg(sumocfg_path, route_path.name)
    return str(sumocfg_path)


def make_time_block_scenario(name, blocks):
    src = Path("simulation/net/intersection.rou.xml")
    route_path = Path(f"simulation/net/{name}.rou.xml")
    sumocfg_path = Path(f"simulation/net/{name}.sumocfg")
    base_tree = ET.parse(src)
    base_root = base_tree.getroot()
    root = ET.Element(base_root.tag, base_root.attrib)
    for route in base_root.findall("route"):
        root.append(deepcopy(route))
    for block_name, begin, end, multipliers in blocks:
        for flow in base_root.findall("flow"):
            block_flow = deepcopy(flow)
            block_flow.set("id", f"{flow.get('id')}_{block_name}")
            block_flow.set("begin", str(begin))
            block_flow.set("end", str(end))
            block_flow.set("vehsPerHour", f"{scaled_rate(flow, **multipliers):.2f}")
            root.append(block_flow)
    tree = ET.ElementTree(root)
    ET.indent(tree, space="    ")
    tree.write(route_path, encoding="UTF-8", xml_declaration=True)
    write_sumocfg(sumocfg_path, route_path.name)
    return str(sumocfg_path)


SCENARIOS = {
    "normal": "simulation/net/intersection.sumocfg",
    "heavy_1_5x": make_scaled_scenario("intersection_heavy_1_5x", all_scale=1.5),
    "heavy_2x": make_scaled_scenario("intersection_heavy_2x", all_scale=2.0),
    "heavy_2_5x": make_scaled_scenario("intersection_heavy_2_5x", all_scale=2.5),
    "imbalanced_ew2x": make_scaled_scenario("intersection_imbalanced_ew2x", ew_scale=2.0),
    "imbalanced_ew3x": make_scaled_scenario("intersection_imbalanced_ew3x", ew_scale=3.0),
    "imbalanced_ew3_5x": make_scaled_scenario("intersection_imbalanced_ew3_5x", ew_scale=3.5),
    "imbalanced_ns2x": make_scaled_scenario("intersection_imbalanced_ns2x", ns_scale=2.0),
    "imbalanced_ns3x": make_scaled_scenario("intersection_imbalanced_ns3x", ns_scale=3.0),
    "imbalanced_ns3_5x": make_scaled_scenario("intersection_imbalanced_ns3_5x", ns_scale=3.5),
}

SCENARIOS["mixed_balanced_peak"] = make_time_block_scenario(
    "intersection_mixed_balanced_peak",
    [("normal", 0, 600, dict(all_scale=1.0)), ("heavy2x", 600, 1200, dict(all_scale=2.0)), ("ew3x", 1200, 1800, dict(ew_scale=3.0)), ("ns3x", 1800, 2400, dict(ns_scale=3.0))],
)
SCENARIOS["mixed_extreme_peak"] = make_time_block_scenario(
    "intersection_mixed_extreme_peak",
    [("normal", 0, 400, dict(all_scale=1.0)), ("heavy25x", 400, 900, dict(all_scale=2.5)), ("ew35x", 900, 1500, dict(ew_scale=3.5)), ("ns35x", 1500, 2100, dict(ns_scale=3.5)), ("normal_tail", 2100, 2400, dict(all_scale=1.0))],
)

SCENARIO_DURATIONS = {name: 2400 if name.startswith("mixed") else 1800 for name in SCENARIOS}

TRAIN_GROUPS = {
    "normal": ["normal"],
    "heavy": ["heavy_1_5x", "heavy_2x", "heavy_2_5x"],
    "ew": ["imbalanced_ew2x", "imbalanced_ew3x", "imbalanced_ew3_5x"],
    "ns": ["imbalanced_ns2x", "imbalanced_ns3x", "imbalanced_ns3_5x"],
    "mixed": ["mixed_balanced_peak", "mixed_extreme_peak"],
}

print("Scenarios:")
for name, path in SCENARIOS.items():
    print(f"{name:<24} {path}")

## 4. Rich traffic-police observation and balanced reward

Base env chỉ trả về 80 grid cells. Rich env thêm các tín hiệu mà người điều tiết thật sẽ nhìn: pha hiện tại, red-time, queue, waiting, max vehicle wait, demand ratio, green age và guard override.

In [ ]:
import numpy as np
from gymnasium import spaces

from common.constants import GRID_CELLS_TOTAL, PHASE_DIRECTIONS, STOPPED_SPEED_THRESHOLD_MPS, PhaseAction
from rl.env.traffic_env import SumoTrafficEnv
from simulation.state.grid_encoder import APPROACH_EDGE_BY_DIRECTION
from simulation.state.waiting_time import total_waiting_time


PHASE_EDGES = {
    phase: {APPROACH_EDGE_BY_DIRECTION[direction] for direction in directions}
    for phase, directions in PHASE_DIRECTIONS.items()
}


def clip01(value):
    return float(max(0.0, min(1.0, value)))


def phase_metrics_from_traci(traci_conn):
    waits = {phase: 0.0 for phase in PhaseAction}
    queues = {phase: 0 for phase in PhaseAction}
    demand = {phase: 0 for phase in PhaseAction}
    max_vehicle_wait = {phase: 0.0 for phase in PhaseAction}
    for vehicle_id in traci_conn.vehicle.getIDList():
        road_id = traci_conn.vehicle.getRoadID(vehicle_id)
        waiting = traci_conn.vehicle.getWaitingTime(vehicle_id)
        speed = traci_conn.vehicle.getSpeed(vehicle_id)
        for phase, edges in PHASE_EDGES.items():
            if road_id in edges:
                waits[phase] += waiting
                demand[phase] += 1
                max_vehicle_wait[phase] = max(max_vehicle_wait[phase], waiting)
                if speed < STOPPED_SPEED_THRESHOLD_MPS:
                    queues[phase] += 1
                break
    return waits, queues, demand, max_vehicle_wait


class PoliceObservationMixin:
    rich_feature_size = 15
    rich_observation_size = GRID_CELLS_TOTAL + rich_feature_size

    def _phase_metrics(self):
        return phase_metrics_from_traci(self._sim.traci)

    def _rich_obs(self, grid):
        waits, queues, demand, max_vehicle_wait = self._phase_metrics()
        ew = PhaseAction.EAST_WEST
        ns = PhaseAction.NORTH_SOUTH
        total_demand = demand[ew] + demand[ns]
        ew_share = demand[ew] / total_demand if total_demand else 0.5
        ns_share = demand[ns] / total_demand if total_demand else 0.5
        current = self._current_phase
        features = np.asarray([
            1.0 if current == ew else 0.0,
            1.0 if current == ns else 0.0,
            clip01(self._red_time_by_phase[ew] / 180.0),
            clip01(self._red_time_by_phase[ns] / 180.0),
            clip01(queues[ew] / 40.0),
            clip01(queues[ns] / 40.0),
            clip01(waits[ew] / 2500.0),
            clip01(waits[ns] / 2500.0),
            clip01(max_vehicle_wait[ew] / 240.0),
            clip01(max_vehicle_wait[ns] / 240.0),
            clip01(getattr(self, "_green_age_s", 0.0) / 120.0),
            1.0 if self._last_action_was_forced else 0.0,
            ew_share,
            ns_share,
            clip01(abs(waits[ew] - waits[ns]) / 2500.0),
        ], dtype=np.float32)
        return np.concatenate([np.asarray(grid, dtype=np.float32), features])


class PoliceTrafficEnv(PoliceObservationMixin, SumoTrafficEnv):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=(self.rich_observation_size,), dtype=np.float32)
        self._green_age_s = 0.0

    def reset(self, *args, **kwargs):
        obs, info = super().reset(*args, **kwargs)
        self._green_age_s = self.green_duration_s
        return self._rich_obs(obs), info

    def step(self, action):
        previous_phase = self._current_phase
        obs, reward, terminated, truncated, info = super().step(action)
        if self._current_phase == previous_phase:
            self._green_age_s += self.green_duration_s
        else:
            self._green_age_s = self.green_duration_s
        return self._rich_obs(obs), reward, terminated, truncated, info


class BalancedPoliceReward:
    def __init__(self):
        self._previous_total = 0.0
        self._previous_arrived = 0

    def reset(self, traci_conn):
        self._previous_total = total_waiting_time(traci_conn)
        self._previous_arrived = 0
        return self._previous_total

    def step(self, traci_conn):
        current_total = total_waiting_time(traci_conn)
        delta_wait = self._previous_total - current_total
        waits, queues, _, max_vehicle_wait = phase_metrics_from_traci(traci_conn)
        wait_values = list(waits.values())
        queue_values = list(queues.values())
        max_phase_wait = max(wait_values) if wait_values else 0.0
        phase_imbalance = abs(waits[PhaseAction.EAST_WEST] - waits[PhaseAction.NORTH_SOUTH])
        total_queue = sum(queue_values)
        max_phase_queue = max(queue_values) if queue_values else 0
        max_wait = max(max_vehicle_wait.values()) if max_vehicle_wait else 0.0
        arrived = traci_conn.simulation.getArrivedNumber()
        reward = (
            1.0 * delta_wait
            + 12.0 * arrived
            - 0.030 * current_total
            - 2.5 * total_queue
            - 6.0 * max_phase_queue
            - 0.10 * max_phase_wait
            - 0.08 * phase_imbalance
            - 0.12 * max_wait
        )
        if max_wait > 180.0:
            reward -= 1.5 * (max_wait - 180.0)
        self._previous_total = current_total
        return reward


print("Rich observation size:", PoliceTrafficEnv.rich_observation_size)

## 5. Dueling Double DQN and scenario-balanced replay

In [ ]:
import random
from collections import defaultdict, deque
from dataclasses import dataclass

import torch
from torch import nn, optim

from common.constants import NUM_ACTIONS
from rl.agent.replay_buffer import Transition


class DuelingDQN(nn.Module):
    def __init__(self, input_size, hidden_size=384, output_size=NUM_ACTIONS):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(input_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.ReLU(),
        )
        self.value = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, 1))
        self.advantage = nn.Sequential(nn.Linear(hidden_size, hidden_size // 2), nn.ReLU(), nn.Linear(hidden_size // 2, output_size))

    def forward(self, x):
        z = self.feature(x)
        value = self.value(z)
        advantage = self.advantage(z)
        return value + advantage - advantage.mean(dim=1, keepdim=True)


class PoliceDQNAgent:
    def __init__(self, input_size, num_actions=NUM_ACTIONS, learning_rate=7.5e-5, gamma=0.995, seed=42, device=None):
        self.num_actions = num_actions
        self.gamma = gamma
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.rng = random.Random(seed)
        self.policy_net = DuelingDQN(input_size=input_size, output_size=num_actions).to(self.device)
        self.target_net = DuelingDQN(input_size=input_size, output_size=num_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.AdamW(self.policy_net.parameters(), lr=learning_rate, weight_decay=1e-5)
        self.loss_fn = nn.SmoothL1Loss()

    def act(self, state, epsilon):
        if self.rng.random() < epsilon:
            return self.rng.randrange(self.num_actions)
        with torch.no_grad():
            state_t = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            return int(self.policy_net(state_t).argmax(dim=1).item())

    def sync_target_network(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def train_step(self, batch):
        states = torch.as_tensor(np.stack([t.state for t in batch]), dtype=torch.float32, device=self.device)
        actions = torch.as_tensor([t.action for t in batch], dtype=torch.int64, device=self.device)
        rewards = torch.as_tensor([t.reward for t in batch], dtype=torch.float32, device=self.device)
        next_states = torch.as_tensor(np.stack([t.next_state for t in batch]), dtype=torch.float32, device=self.device)
        dones = torch.as_tensor([t.done for t in batch], dtype=torch.float32, device=self.device)
        q_values = self.policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_actions = self.policy_net(next_states).argmax(dim=1, keepdim=True)
            next_q = self.target_net(next_states).gather(1, next_actions).squeeze(1)
            targets = rewards + self.gamma * next_q * (1.0 - dones)
        loss = self.loss_fn(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=10.0)
        self.optimizer.step()
        return float(loss.item())


class ScenarioReplayBuffer:
    def __init__(self, capacity_per_group=35000, seed=42):
        self.buffers = {group: deque(maxlen=capacity_per_group) for group in TRAIN_GROUPS}
        self.rng = random.Random(seed)

    def push(self, group, transition):
        self.buffers[group].append(transition)

    def __len__(self):
        return sum(len(buf) for buf in self.buffers.values())

    def sample(self, batch_size, group_weights):
        batch = []
        groups = list(group_weights)
        weights = [group_weights[g] for g in groups]
        attempts = 0
        while len(batch) < batch_size and attempts < batch_size * 20:
            group = self.rng.choices(groups, weights=weights, k=1)[0]
            if self.buffers[group]:
                batch.append(self.rng.choice(list(self.buffers[group])))
            attempts += 1
        if len(batch) < batch_size:
            all_items = [item for buf in self.buffers.values() for item in buf]
            batch.extend(self.rng.sample(all_items, batch_size - len(batch)))
        return batch


def save_police_checkpoint(path, agent, episode, extra=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "episode": episode,
        "input_size": PoliceTrafficEnv.rich_observation_size,
        "architecture": "dueling_double_dqn_rich_police_observation",
        "policy_state_dict": agent.policy_net.state_dict(),
        "target_state_dict": agent.target_net.state_dict(),
        "optimizer_state_dict": agent.optimizer.state_dict(),
        "extra": extra or {},
    }, path)


def load_police_checkpoint(path, agent):
    checkpoint = torch.load(Path(path), map_location=agent.device)
    agent.policy_net.load_state_dict(checkpoint["policy_state_dict"])
    agent.target_net.load_state_dict(checkpoint["target_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    return int(checkpoint["episode"])


print("Device ready:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

## 6. Full interleaved training from scratch

Curriculum chuyển từ ổn định cơ bản sang peak/imbalanced, rồi tự tăng tỉ lệ nhóm đang yếu dựa trên rolling mean wait.

In [ ]:
from tqdm.auto import tqdm
from benchmark.metrics import EpisodeMetrics

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints_police_full")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 20260728
FULL_TRAIN_EPISODES = 5000
BATCH_SIZE = 128
MIN_REPLAY_SIZE = 6000
TRAIN_EVERY_N_STEPS = 4
TARGET_SYNC_EVERY_EPISODES = 8
CHECKPOINT_EVERY_EPISODES = 100

GUARD_CONFIG = dict(
    max_red_time_s=None,
    soft_red_time_s=75.0,
    hard_red_time_s=125.0,
    starving_queue_threshold=4,
    starving_wait_time_s=150.0,
)


def resolve_backend():
    try:
        import libsumo  # noqa: F401
        return "libsumo"
    except ImportError:
        return "traci"


def group_weights_for_episode(episode, rolling_waits):
    if episode <= 1000:
        weights = {"normal": 0.25, "heavy": 0.30, "ew": 0.15, "ns": 0.15, "mixed": 0.15}
    elif episode <= 3500:
        weights = {"normal": 0.15, "heavy": 0.25, "ew": 0.20, "ns": 0.20, "mixed": 0.20}
    else:
        weights = {"normal": 0.12, "heavy": 0.22, "ew": 0.22, "ns": 0.22, "mixed": 0.22}
        known = {g: (sum(v) / len(v)) for g, v in rolling_waits.items() if len(v) >= 8}
        if known:
            worst = max(known, key=known.get)
            weights[worst] += 0.08
            scale = sum(weights.values())
            weights = {k: v / scale for k, v in weights.items()}
    return weights


def epsilon_for_episode(episode):
    if episode <= 3500:
        return 1.0 + min(1.0, episode / 3500) * (0.05 - 1.0)
    return 0.03


def sample_group_and_scenario(rng, group_weights):
    groups = list(group_weights)
    weights = [group_weights[g] for g in groups]
    group = rng.choices(groups, weights=weights, k=1)[0]
    return group, rng.choice(TRAIN_GROUPS[group])


def make_env(scenario_name, backend):
    env = PoliceTrafficEnv(
        sumocfg_path=SCENARIOS[scenario_name],
        episode_duration_s=SCENARIO_DURATIONS[scenario_name],
        green_duration_s=10.0,
        backend=backend,
        **GUARD_CONFIG,
    )
    env._reward_engine = BalancedPoliceReward()
    return env


def run_training_episode(env, agent, buffer, group, epsilon, episode_seed, global_step, replay_weights):
    obs, info = env.reset(seed=episode_seed)
    metrics = EpisodeMetrics()
    metrics.record_step(info["total_waiting_time_s"], int(obs[:GRID_CELLS_TOTAL].sum()), info["sim_time_s"])
    episode_reward = 0.0
    loss_sum = 0.0
    loss_count = 0
    forced_count = 0
    arrived_vehicles = info["arrived_vehicles"]
    terminated = truncated = False
    while not (terminated or truncated):
        action = agent.act(obs, epsilon)
        next_obs, reward, terminated, truncated, info = env.step(action)
        buffer.push(group, Transition(obs, action, reward, next_obs, terminated or truncated))
        obs = next_obs
        episode_reward += reward
        arrived_vehicles = info["arrived_vehicles"]
        forced_count += int(info.get("action_forced_by_guard", False))
        metrics.record_step(info["total_waiting_time_s"], int(obs[:GRID_CELLS_TOTAL].sum()), info["sim_time_s"])
        global_step += 1
        if len(buffer) >= MIN_REPLAY_SIZE and global_step % TRAIN_EVERY_N_STEPS == 0:
            loss_sum += agent.train_step(buffer.sample(BATCH_SIZE, replay_weights))
            loss_count += 1
    final_metrics = metrics.finalize(arrived_vehicles=arrived_vehicles).to_dict()
    final_metrics["forced_action_rate"] = forced_count / max(1, metrics.total_steps)
    return final_metrics, episode_reward, (loss_sum / loss_count if loss_count else 0.0), global_step


rng = random.Random(RANDOM_SEED)
backend = resolve_backend()
agent = PoliceDQNAgent(input_size=PoliceTrafficEnv.rich_observation_size, seed=RANDOM_SEED)
buffer = ScenarioReplayBuffer(capacity_per_group=40000, seed=RANDOM_SEED)
rolling_waits = defaultdict(lambda: deque(maxlen=40))
global_step = 0

print("Backend:", backend)
progress = tqdm(range(1, FULL_TRAIN_EPISODES + 1), desc="full police training", unit="ep")
for episode in progress:
    train_weights = group_weights_for_episode(episode, rolling_waits)
    replay_weights = {"normal": 0.15, "heavy": 0.25, "ew": 0.20, "ns": 0.20, "mixed": 0.20}
    group, scenario_name = sample_group_and_scenario(rng, train_weights)
    epsilon = epsilon_for_episode(episode)
    env = make_env(scenario_name, backend)
    try:
        metrics, episode_reward, avg_loss, global_step = run_training_episode(
            env, agent, buffer, group, epsilon, RANDOM_SEED + episode, global_step, replay_weights
        )
    finally:
        env.close()
    rolling_waits[group].append(metrics["mean_waiting_time_s"])
    if episode % TARGET_SYNC_EVERY_EPISODES == 0:
        agent.sync_target_network()
    if episode % CHECKPOINT_EVERY_EPISODES == 0 or episode == FULL_TRAIN_EPISODES:
        save_police_checkpoint(CHECKPOINT_DIR / f"dqn_police_episode_{episode}.pt", agent, episode, {"guard": GUARD_CONFIG})
    progress.set_postfix(group=group, scenario=scenario_name, eps=f"{epsilon:.2f}", wait=f"{metrics['mean_waiting_time_s']:.0f}", forced=f"{metrics['forced_action_rate']:.2f}", loss=f"{avg_loss:.2f}", replay=len(buffer))

save_police_checkpoint(CHECKPOINT_DIR / "dqn_police_final.pt", agent, FULL_TRAIN_EPISODES, {"guard": GUARD_CONFIG})
print("Saved:", CHECKPOINT_DIR / "dqn_police_final.pt")
print("Recent group mean waits:")
for group, waits in sorted(rolling_waits.items()):
    print(f"{group:<8} {sum(waits) / len(waits):8.2f} n={len(waits)}")

## 7. Hard guardrail evaluation and checkpoint selection

In [ ]:
import json
import shutil
from datetime import datetime, timezone

from benchmark.policies import FixedTimePolicy

EVAL_SEEDS = [101, 102, 103, 104, 105]
EVAL_SCENARIOS = ["normal", "heavy_2x", "imbalanced_ew3x", "imbalanced_ns3x", "mixed_balanced_peak", "mixed_extreme_peak"]


class PolicePolicy:
    def __init__(self, agent):
        self.agent = agent
    def reset(self):
        pass
    def select_action(self, obs, sim_time_s):
        return self.agent.act(obs, 0.0)


def run_policy_episode(env, policy, seed):
    policy.reset()
    obs, info = env.reset(seed=seed)
    metrics = EpisodeMetrics()
    metrics.record_step(info["total_waiting_time_s"], int(obs[:GRID_CELLS_TOTAL].sum()), info["sim_time_s"])
    forced_count = 0
    arrived = info["arrived_vehicles"]
    terminated = truncated = False
    while not (terminated or truncated):
        action = policy.select_action(obs, info["sim_time_s"])
        obs, reward, terminated, truncated, info = env.step(action)
        forced_count += int(info.get("action_forced_by_guard", False))
        arrived = info["arrived_vehicles"]
        metrics.record_step(info["total_waiting_time_s"], int(obs[:GRID_CELLS_TOTAL].sum()), info["sim_time_s"])
    result = metrics.finalize(arrived).to_dict()
    result["forced_action_rate"] = forced_count / max(1, metrics.total_steps)
    return result


def aggregate(runs):
    return {key: sum(run[key] for run in runs) / len(runs) for key in runs[0]}


def improvement_pct(fixed, dqn):
    return {
        "mean_wait": (fixed["mean_waiting_time_s"] - dqn["mean_waiting_time_s"]) / fixed["mean_waiting_time_s"] * 100 if fixed["mean_waiting_time_s"] else 0.0,
        "final_wait": (fixed["final_waiting_time_s"] - dqn["final_waiting_time_s"]) / fixed["final_waiting_time_s"] * 100 if fixed["final_waiting_time_s"] else 0.0,
        "queue": (fixed["mean_queue_length"] - dqn["mean_queue_length"]) / fixed["mean_queue_length"] * 100 if fixed["mean_queue_length"] else 0.0,
        "arrived": (dqn["arrived_vehicles"] - fixed["arrived_vehicles"]) / fixed["arrived_vehicles"] * 100 if fixed["arrived_vehicles"] else 0.0,
        "forced_action_rate": dqn.get("forced_action_rate", 0.0),
    }


def eval_scenario_with_policy(scenario_name, policy, seed):
    env = make_env(scenario_name, backend)
    try:
        return run_policy_episode(env, policy, seed)
    finally:
        env.close()


def guardrail_failures(improvements, per_seed_dqn, per_seed_imp):
    failures = []
    required_mean = {
        "normal": 5.0,
        "heavy_2x": 8.0,
        "imbalanced_ew3x": 0.0,
        "imbalanced_ns3x": 10.0,
        "mixed_balanced_peak": 55.0,
        "mixed_extreme_peak": 35.0,
    }
    for scenario, min_imp in required_mean.items():
        if improvements[scenario]["mean_wait"] < min_imp:
            failures.append(f"{scenario}_mean_wait_below_{min_imp}")
    if improvements["heavy_2x"]["arrived"] < -6.0:
        failures.append("heavy_throughput_drop")
    for scenario, imp in improvements.items():
        if imp["forced_action_rate"] > 0.18:
            failures.append(f"{scenario}_too_many_guard_overrides")
    max_mean_wait = {"normal": 160, "heavy_2x": 850, "imbalanced_ew3x": 700, "imbalanced_ns3x": 700, "mixed_balanced_peak": 850, "mixed_extreme_peak": 950}
    max_final_wait = {"normal": 500, "heavy_2x": 2500, "imbalanced_ew3x": 2500, "imbalanced_ns3x": 2500, "mixed_balanced_peak": 3000, "mixed_extreme_peak": 3500}
    for scenario, runs in per_seed_dqn.items():
        for seed, metrics, imp in zip(EVAL_SEEDS, runs, per_seed_imp[scenario]):
            if metrics["mean_waiting_time_s"] > max_mean_wait[scenario]:
                failures.append(f"{scenario}_seed_{seed}_mean_wait_spike")
            if metrics["final_waiting_time_s"] > max_final_wait[scenario]:
                failures.append(f"{scenario}_seed_{seed}_final_wait_spike")
            if scenario == "imbalanced_ns3x" and imp["mean_wait"] < 0.0:
                failures.append(f"ns3x_seed_{seed}_worse_than_fixed")
            if scenario == "imbalanced_ew3x" and imp["mean_wait"] < -5.0:
                failures.append(f"ew3x_seed_{seed}_too_much_regression")
    return failures


def police_score(improvements):
    return sum(
        3.0 * imp["mean_wait"] + 1.5 * imp["queue"] + 1.0 * imp["arrived"] - 80.0 * imp["forced_action_rate"]
        for imp in improvements.values()
    )


fixed_runs = {}
fixed_agg = {}
print("Fixed-time baselines")
for scenario in EVAL_SCENARIOS:
    fixed_policy = FixedTimePolicy(green_duration_s=20.0)
    fixed_runs[scenario] = [eval_scenario_with_policy(scenario, fixed_policy, seed) for seed in EVAL_SEEDS]
    fixed_agg[scenario] = aggregate(fixed_runs[scenario])
    print(scenario, fixed_agg[scenario])

candidate_paths = sorted(CHECKPOINT_DIR.glob("dqn_police_episode_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))[-30:]
if (CHECKPOINT_DIR / "dqn_police_final.pt").exists():
    candidate_paths.append(CHECKPOINT_DIR / "dqn_police_final.pt")

results = []
for path in candidate_paths:
    eval_agent = PoliceDQNAgent(input_size=PoliceTrafficEnv.rich_observation_size, seed=RANDOM_SEED)
    trained_episode = load_police_checkpoint(path, eval_agent)
    policy = PolicePolicy(eval_agent)
    dqn_runs = {}
    dqn_agg = {}
    improvements = {}
    per_seed_imp = {}
    for scenario in EVAL_SCENARIOS:
        dqn_runs[scenario] = [eval_scenario_with_policy(scenario, policy, seed) for seed in EVAL_SEEDS]
        dqn_agg[scenario] = aggregate(dqn_runs[scenario])
        improvements[scenario] = improvement_pct(fixed_agg[scenario], dqn_agg[scenario])
        per_seed_imp[scenario] = [improvement_pct(f, d) for f, d in zip(fixed_runs[scenario], dqn_runs[scenario])]
    failures = guardrail_failures(improvements, dqn_runs, per_seed_imp)
    passed = len(failures) == 0
    score = police_score(improvements)
    results.append(dict(path=str(path), episode=trained_episode, passed=passed, score=score, improvements=improvements, aggregated=dqn_agg, failures=failures))
    print(f"\n{path.name:<30} ep={trained_episode:<5} pass={passed} score={score:8.2f}")
    if failures:
        print("  failures:", ", ".join(failures[:10]), "..." if len(failures) > 10 else "")
    for scenario, imp in improvements.items():
        print(f"  {scenario:<22} wait={imp['mean_wait']:7.1f}% queue={imp['queue']:7.1f}% arrived={imp['arrived']:7.1f}% forced={imp['forced_action_rate']:.2f}")

passed = [r for r in results if r["passed"]]
selected = max(passed, key=lambda r: r["score"]) if passed else max(results, key=lambda r: r["score"])
selected_path = CHECKPOINT_DIR / "dqn_police_best.pt"
shutil.copy2(selected["path"], selected_path)

report_path = CHECKPOINT_DIR / f"police_eval_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
report_path.write_text(json.dumps({"selected": selected, "results": results}, indent=2), encoding="utf-8")
print("\nSelected:", selected["path"], "passed=", selected["passed"], "score=", selected["score"])
print("Wrote:", selected_path)
print("Report:", report_path)